In [1]:
import numpy as np
import pandas as pd
import plotly.express as px

In [2]:
FWD_MS_COL = "forward_ms"
BWD_MS_COL = "backward_ms"
FWD_MEM_COL = "forward_memory_mb"
BWD_MEM_COL = "backward_memory_mb"
MS_TOTAL = "ms_total"

CONV_COL = "conv_type"
DATASET_COL = "dataset"
BACKEND_COL = "backend"
HEADS_COL = "heads"
FEATURE_DIM_COL = "feature_dim"

In [3]:
REORDERING_COL = "graph_reordering_partition_size"

# other columns used for tuning parameters

In [4]:
BASELINE = "dgl"

def add_speedup_columns(df_in: pd.DataFrame, baseline: str = BASELINE) -> pd.DataFrame:
    """
    For each (dataset, feature_dim, heads) triple, compute speedup
    vs `baseline` separately for forward_ms and backward_ms.
    """
    df = df_in.copy()
    idx_cols = [CONV_COL, DATASET_COL, FEATURE_DIM_COL, HEADS_COL, REORDERING_COL]

    for metric in [FWD_MS_COL, BWD_MS_COL, FWD_MEM_COL, BWD_MEM_COL, MS_TOTAL]:
        base_col = f"{metric}_baseline_{baseline}"
        speed_col = f"{metric}_speedup_vs_{baseline}"
        df[base_col] = np.nan
        df[speed_col] = np.nan

        for key, group in df.groupby(idx_cols):
            base_rows = group[group["backend"] == baseline]
            if base_rows.empty:
                continue
            base_val = base_rows.iloc[0][metric]
            if not np.isfinite(base_val) or base_val <= 0:
                continue

            mask = (
                (df[CONV_COL] == key[0]) &
                (df[DATASET_COL] == key[1]) &
                (df[FEATURE_DIM_COL] == key[2]) &
                (df[HEADS_COL] == key[3]) &
                (df[REORDERING_COL] == key[4])
            )
            df.loc[mask, base_col] = base_val
            df.loc[mask, speed_col] = base_val / df.loc[mask, metric]

    return df


In [5]:
gcn_df = pd.concat([
    pd.read_csv("/home/fvelikon/projects/cuda_exp/out/gcn_no_reordering", index_col=0),
    pd.read_csv("/home/fvelikon/projects/cuda_exp/out/gcn_torch_native", index_col=0),
    pd.read_csv("/home/fvelikon/projects/cuda_exp/out/gcn_cusoarse_with_precompute_bwd", index_col=0),
    pd.read_csv("/home/fvelikon/projects/cuda_exp/out/gcn_partition", index_col=0),
])
gcn_df[HEADS_COL] = 1

# min_aggr_new_fp32
# MIN AGGR:
min_aggr_df = pd.concat([
    pd.read_csv("/home/fvelikon/projects/cuda_exp/out/min_aggr_no_reordering", index_col=0),
    pd.read_csv("/home/fvelikon/projects/cuda_exp/out/min_aggr_partition", index_col=0),
    pd.read_csv("/home/fvelikon/projects/cuda_exp/out/min_aggr_new_fp32", index_col=0),
    pd.read_csv("/home/fvelikon/projects/cuda_exp/out/min_aggr_proper_sweep", index_col=0),
])
min_aggr_df[HEADS_COL] = 1

#gatv2

gatv2_df = pd.concat([
    pd.read_csv("/home/fvelikon/projects/cuda_exp/out/test_graph_reordering", index_col=0),
    pd.read_csv("/home/fvelikon/projects/cuda_exp/out/gat_v2_no_reordering", index_col=0),
    pd.read_csv("/home/fvelikon/projects/cuda_exp/out/gatv2_partition", index_col=0),
    pd.read_csv("/home/fvelikon/projects/cuda_exp/out/gat_v2_cuda_new_1", index_col=0),
    pd.read_csv("/home/fvelikon/projects/cuda_exp/out/gat_v2_cuda_new_2", index_col=0),
    pd.read_csv("/home/fvelikon/projects/cuda_exp/out/gat_v2_cuda_new_3", index_col=0),
    pd.read_csv("/home/fvelikon/projects/cuda_exp/out/gatv2_new_dims_cuda_dgl", index_col=0),
    pd.read_csv("/home/fvelikon/projects/cuda_exp/out/gatv2_new_dims_cuda_dgl_pyg_dim_128", index_col=0),
    pd.read_csv("/home/fvelikon/projects/cuda_exp/out/gatv2_new_dims_pyg", index_col=0),
])

gatv2_df = gatv2_df[gatv2_df[BACKEND_COL] != "cugraph"]
gatv2_df = gatv2_df[gatv2_df[FEATURE_DIM_COL] != 512]


# graph transformer
gt_df = pd.concat([
    pd.read_csv("/home/fvelikon/projects/cuda_exp/out/gt_new", index_col=0),
    pd.read_csv("/home/fvelikon/projects/cuda_exp/out/gt_cuda_new", index_col=0),
    pd.read_csv("/home/fvelikon/projects/cuda_exp/out/gt_partition", index_col=0),
    pd.read_csv("/home/fvelikon/projects/cuda_exp/out/gt_dfgnn", index_col=0),
])

gt_df = gt_df[gt_df[BACKEND_COL] != "cugraph"]


df = pd.concat([
    gcn_df,
    min_aggr_df,
    gatv2_df,
    gt_df,
])

In [6]:
df[MS_TOTAL] = df[FWD_MS_COL] + df[BWD_MS_COL]
df = add_speedup_columns(df, BASELINE)
df["density"] = df["num_edges"] / df["num_nodes"] / (df["num_nodes"] - 1)
df["average_node_degree"] = df["num_edges"] / df["num_nodes"]


In [7]:
def build_speedup_table_with_dataset_index(
    df_speed: pd.DataFrame,
    baseline: str,
) -> pd.DataFrame:
    """
    Build a single speedup table with a MultiIndex including dataset.

    Index:
        ('conv_type', 'dataset', 'heads', 'feature_dim')  if 'heads' in df_speed.columns
        ('conv_type', 'dataset', 'feature_dim')          otherwise

    Columns:
        MultiIndex (metric_short, backend), where metric_short ∈
        {'Fwd time', 'Bwd time', 'Fwd mem', 'Bwd mem'} and values are
        speedup vs baseline (>1 = better than baseline).

    Baseline backend column is dropped (speedup is always 1.0).
    """
    metrics = [
        (FWD_MS_COL,  "Fwd time"),
        (BWD_MS_COL,  "Bwd time"),
        (FWD_MEM_COL, "Fwd mem"),
        (BWD_MEM_COL, "Bwd mem"),
        (MS_TOTAL, "Fwd + Bwd time")
    ]

    # decide index cols
    idx_cols = [CONV_COL, DATASET_COL, HEADS_COL, FEATURE_DIM_COL]

    metric_frames = []

    for metric, short_name in metrics:
        speed_col = f"{metric}_speedup_vs_{baseline}"
        if speed_col not in df_speed.columns:
            continue

        df_metric = df_speed[np.isfinite(df_speed[speed_col])].copy()
        if df_metric.empty:
            continue

        pivot = df_metric.pivot_table(
            index=idx_cols,
            columns=BACKEND_COL,
            values=speed_col,
            aggfunc="max",
        )

        # drop baseline backend (always 1.0)
        pivot = pivot.drop(columns=[baseline], errors="ignore")

        if pivot.empty:
            continue

        # attach metric name as top-level column
        pivot.columns = pd.MultiIndex.from_product(
            [[short_name], pivot.columns],
            names=["metric", "backend"],
        )

        metric_frames.append(pivot)

    if not metric_frames:
        return pd.DataFrame()

    table = pd.concat(metric_frames, axis=1)

    # sort rows & columns
    table = table.sort_index()
    table = table.sort_index(axis=1)

    return table

def build_raw_table_with_dataset_index(
    df: pd.DataFrame,
    index = None,
) -> pd.DataFrame:
    """
    Build a raw-metrics table with the same structure as
    `build_speedup_table_with_dataset_index`, but containing
    *raw* runtimes (ms) and memory (MB) for all backends.

    Index:
        (conv_type, dataset, heads, feature_dim)

    Columns:
        MultiIndex (metric_short, backend), where metric_short ∈
        {'Fwd time', 'Bwd time', 'Fwd mem', 'Bwd mem', 'Fwd + Bwd time'}.

    Values:
        Raw numbers from the corresponding metric columns.
    """
    metrics = [
        (FWD_MS_COL,  "Fwd time"),
        (BWD_MS_COL,  "Bwd time"),
        (FWD_MEM_COL, "Fwd mem"),
        (BWD_MEM_COL, "Bwd mem"),
        (MS_TOTAL,    "Fwd + Bwd time"),
    ]

    idx_cols = index or [CONV_COL, DATASET_COL, HEADS_COL, FEATURE_DIM_COL]

    metric_frames: list[pd.DataFrame] = []

    for metric, short_name in metrics:
        if metric not in df.columns:
            continue

        df_metric = df[np.isfinite(df[metric])].copy()
        if df_metric.empty:
            continue

        pivot = df_metric.pivot_table(
            index=idx_cols,
            columns=BACKEND_COL,
            values=metric,
            aggfunc="min",  # same choice as in speedup table
        )

        if pivot.empty:
            continue

        # put metric name on top level
        pivot.columns = pd.MultiIndex.from_product(
            [[short_name], pivot.columns],
            names=["metric", "backend"],
        )

        metric_frames.append(pivot)

    if not metric_frames:
        return pd.DataFrame()

    table = pd.concat(metric_frames, axis=1)

    table = table.sort_index()
    table = table.sort_index(axis=1)

    return table


In [8]:
no_reordering_df = df[df[REORDERING_COL] == -1]
[
    [
        CONV_COL,
        FWD_MS_COL,
        BWD_MS_COL,
        FWD_MEM_COL,
        BWD_MEM_COL,
        DATASET_COL,
        BACKEND_COL,
        HEADS_COL,
        FEATURE_DIM_COL,
        REORDERING_COL,
    ]
]
reordering_df = df[df[REORDERING_COL] != -1]

[
    [
        CONV_COL,
        FWD_MS_COL,
        BWD_MS_COL,
        FWD_MEM_COL,
        BWD_MEM_COL,
        DATASET_COL,
        BACKEND_COL,
        HEADS_COL,
        FEATURE_DIM_COL,
        REORDERING_COL,
    ]
]

[['conv_type',
  'forward_ms',
  'backward_ms',
  'forward_memory_mb',
  'backward_memory_mb',
  'dataset',
  'backend',
  'heads',
  'feature_dim',
  'graph_reordering_partition_size']]

In [9]:
datasets_df = df.groupby(DATASET_COL)[["num_nodes", "num_edges"]].max().reset_index()
datasets_df["density"] = datasets_df["num_edges"] / datasets_df["num_nodes"] / (datasets_df["num_nodes"] - 1)
datasets_df["average_node_degree"] = datasets_df["num_edges"] / datasets_df["num_nodes"]

datasets_df = datasets_df.sort_values(by="density", ascending=False).reset_index(drop=True)

In [10]:
# datasets_df["dataset"].values.tolist()

In [11]:
print(datasets_df.to_markdown())

|    | dataset       |   num_nodes |   num_edges |     density |   average_node_degree |
|---:|:--------------|------------:|------------:|------------:|----------------------:|
|  0 | hm-prices     |       46563 |    21461990 | 0.00989914  |             460.924   |
|  1 | hm-categories |       46563 |    21461990 | 0.00989914  |             460.924   |
|  2 | tolokers-2    |       11758 |     1038000 | 0.00750875  |              88.2803  |
|  3 | avazu-ctr     |       76269 |    21968154 | 0.00377662  |             288.035   |
|  4 | cora          |        2708 |       10556 | 0.00144     |               3.89808 |
|  5 | citeseer      |        3327 |        9104 | 0.00082273  |               2.7364  |
|  6 | twitch-views  |      168114 |    13595114 | 0.000481036 |              80.8684  |
|  7 | pubmed        |       19717 |       88648 | 0.000228039 |               4.49602 |
|  8 | artnet-views  |       50405 |      560696 | 0.000220693 |              11.1238  |
|  9 | artnet-exp    

In [12]:
datasets_df

,dataset,num_nodes,num_edges,density,average_node_degree
0,hm-prices,46563,21461990,0.009899,460.923695
1,hm-categories,46563,21461990,0.009899,460.923695
2,tolokers-2,11758,1038000,0.007509,88.280320
3,avazu-ctr,76269,21968154,0.003777,288.035165
4,cora,2708,10556,0.001440,3.898080
5,citeseer,3327,9104,0.000823,2.736399
6,twitch-views,168114,13595114,0.000481,80.868423
7,pubmed,19717,88648,0.000228,4.496019
8,artnet-views,50405,560696,0.000221,11.123817
9,artnet-exp,50405,560696,0.000221,11.123817


In [13]:
datasets_subset = [
    "tolokers-2",
    "hm-categories",
    "artnet-views",
    "city-roads-M",
    "avazu-ctr",
    "city-reviews",
    "twitch-views",
    "ogbn-arxiv",
    "ogbn-products",
]

In [14]:
# df = df[df[DATASET_COL].isin(datasets_subset)]

In [15]:
px.box(df[(df[CONV_COL] == "gt") & (df[BACKEND_COL] != "dgl")], x=BACKEND_COL, y="ms_total_speedup_vs_dgl")

In [16]:
px.box(df[(df[CONV_COL] == "gt") & (df[BACKEND_COL] != "dgl")], x=BACKEND_COL, y="forward_ms_speedup_vs_dgl")

In [17]:
px.box(df[(df[CONV_COL] == "gcn") & (df[BACKEND_COL] != "dgl")], x=BACKEND_COL, y="forward_ms_speedup_vs_dgl")

In [18]:
px.box(df[(df[CONV_COL] == "gat_v2") & (df[BACKEND_COL] != "dgl")], x=BACKEND_COL, y="forward_ms_speedup_vs_dgl")

In [19]:
px.box(df[(df[CONV_COL] == "min_aggr") & (df[BACKEND_COL] != "dgl")], x=BACKEND_COL, y="ms_total_speedup_vs_dgl")

In [20]:
from scipy.stats import gmean

In [21]:
df = df[
    ~df[DATASET_COL].isin([
        "web-fraud",
        "web-topics",
        "artnet-views",
        "hm-prices"
    ])
]

In [22]:
gt_speedup = build_speedup_table_with_dataset_index(df[df[CONV_COL] == "gt"], "dgl")

In [23]:
gatv2_speedup = build_speedup_table_with_dataset_index(df[df[CONV_COL] == "gat_v2"], "dgl")

In [24]:
min_aggr_speedup = build_speedup_table_with_dataset_index(df[df[CONV_COL] == "min_aggr"], "dgl")

In [25]:
g_speedup = build_speedup_table_with_dataset_index(df[df[CONV_COL] == "gcn"], "dgl")
# g_speedup

In [26]:
gt_speedup.to_csv("gt_speedup.csv")
gatv2_speedup.to_csv("gatv2_speedup.csv")
min_aggr_speedup.to_csv("min_aggr_speedup.csv")
g_speedup.to_csv("gcn_speedup.csv")


In [27]:
# min_aggr_speedup.tail(60)

In [28]:
# min_aggr_speedup[min_aggr_speedup > 0].apply(gmean, nan_policy='omit')

In [29]:
# tuples = [
#             ('Fwd + Bwd time',                 'cusparse'),
#             ('Fwd + Bwd time', 'cusparse_precomputed_bwd'),
#             ('Fwd + Bwd time',                  'fusegnn'),
#             ('Fwd + Bwd time',                      'pyg'),
#             ('Fwd + Bwd time',                    'tcgnn'),
#             ('Fwd + Bwd time',         'torch_native_gcn'),
#             ('Fwd + Bwd time',      'triton_block_sparse'),

#           ]
# columns = pd.MultiIndex.from_tuples(tuples, names=['first', 'second'])
# g_speedup[columns].tail(60)

In [30]:
# g_speedup[g_speedup > 0].apply(gmean, nan_policy='omit')

In [31]:
import os
import re
from pathlib import Path
import pandas as pd
import numpy as np

# ----------------------------
# Utils
# ----------------------------
METRICS_KEEP = {"Fwd time", "Bwd time", "Fwd mem", "Bwd mem"}  # ignore "Fwd + Bwd time"

def latex_escape(s: str) -> str:
    # minimal escaping for dataset/backends containing underscores, etc.
    if s is None:
        return ""
    s = str(s)
    repl = {
        "\\": r"\textbackslash{}",
        "_": r"\_",
        "%": r"\%",
        "&": r"\&",
        "#": r"\#",
        "{": r"\{",
        "}": r"\}",
        "$": r"\$",
        "~": r"\textasciitilde{}",
        "^": r"\textasciicircum{}",
    }
    for k, v in repl.items():
        s = s.replace(k, v)
    return s

def fmt_num(x, ndigits=2):
    if pd.isna(x):
        return "--"
    try:
        x = float(x)
    except Exception:
        return str(x)
    # keep very small values readable
    if x != 0.0 and abs(x) < 1e-3:
        return f"{x:.2e}"
    return f"{x:.{ndigits}f}"

def build_latex_table_for_setup(df_long: pd.DataFrame,
                                conv_type: str,
                                heads: int,
                                feature_dim: int,
                                backend: str,
                                caption: str,
                                label: str,
                                ndigits=2) -> str:
    """
    Produce LaTeX table (tabular) with:
      rows: dataset
      cols: Fwd time, Bwd time, Fwd mem, Bwd mem
    """
    sub = df_long[
        (df_long["conv_type"] == conv_type) &
        (df_long["heads"] == heads) &
        (df_long["feature_dim"] == feature_dim) &
        (df_long["backend"] == backend) &
        (df_long["metric"].isin(METRICS_KEEP))
    ].copy()

    # pivot: dataset x metric
    piv = sub.pivot_table(index="dataset", columns="metric", values="value", aggfunc="first")
    # enforce column order
    col_order = ["Fwd time", "Bwd time", "Fwd mem", "Bwd mem"]
    for c in col_order:
        if c not in piv.columns:
            piv[c] = np.nan
    piv = piv[col_order].sort_index()

    # format rows
    rows = []
    for ds, row in piv.iterrows():
        ds_tex = r"\texttt{" + latex_escape(ds) + "}"
        vals = [fmt_num(row[c], ndigits=ndigits) for c in col_order]
        rows.append(ds_tex + " & " + " & ".join(vals) + r" \\")
    body = "\n".join(rows)

    # header
    out = rf"""\begin{{table}}[H]
\centering
\small
\begin{{tabular}}{{lcccc}}
\toprule
Dataset & Fwd time & Bwd time & Fwd mem & Bwd mem \\
\midrule
{body}
\bottomrule
\end{{tabular}}
\caption{{{caption}}}
\label{{{label}}}
\end{{table}}
"""
    return out

def export_all_tables(csv_path: str,
                      out_dir: str,
                      conv_display_name: str,
                      backends: list[str] | None = None,
                      ndigits=2):
    df_long = parse_speedup_csv(csv_path)

    Path(out_dir).mkdir(parents=True, exist_ok=True)

    # available backends in this csv
    avail_backends = sorted(df_long["backend"].dropna().unique().tolist())
    if backends is None:
        backends = avail_backends
    else:
        backends = [b for b in backends if b in avail_backends]

    # iterate setups
    setups = (
        df_long[["conv_type", "heads", "feature_dim"]]
        .dropna()
        .drop_duplicates()
        .sort_values(["conv_type", "heads", "feature_dim"])
        .itertuples(index=False)
    )

    index_lines = []
    for conv_type, heads, feat in setups:
        for backend in backends:
            caption = (f"{conv_display_name} speedups on all datasets "
                       f"(backend={backend}, heads={int(heads)}, feature_dim={int(feat)}). "
                       f"Values are relative to DGL: time columns are latency speedups, "
                       f"memory columns are peak-memory ratios (higher is better).")
            label = f"tab:{conv_type}_{backend}_h{int(heads)}_d{int(feat)}"
            tex = build_latex_table_for_setup(
                df_long=df_long,
                conv_type=str(conv_type),
                heads=int(heads),
                feature_dim=int(feat),
                backend=str(backend),
                caption=latex_escape(caption),
                label=latex_escape(label),
                ndigits=ndigits
            )
            fname = f"{conv_type}__{backend}__h{int(heads)}__d{int(feat)}.tex"
            fpath = Path(out_dir) / fname
            fpath.write_text(tex)

            index_lines.append(rf"\input{{{fpath.as_posix()}}}")

    # write a master include file
    (Path(out_dir) / "ALL_TABLES.tex").write_text("\n".join(index_lines))
    print(f"[OK] Wrote {len(index_lines)} tables into {out_dir}")
    print(f"[OK] Master include: {Path(out_dir) / 'ALL_TABLES.tex'}")
    print(f"[INFO] Available backends: {avail_backends}")


In [32]:
import pandas as pd
import numpy as np
import re

METRICS_KEEP = {"Fwd time", "Bwd time", "Fwd mem", "Bwd mem"}  # ignore "Fwd + Bwd time"
ID_COLS = ["conv_type", "dataset", "heads", "feature_dim"]

_tuple_pat = re.compile(r"^\(\s*(.+?)\s*,\s*(.+?)\s*\)$")

def _maybe_make_multiindex(cols):
    """
    If cols are strings like "(Bwd mem, cusparse)" -> convert to MultiIndex (metric, backend).
    If already MultiIndex -> keep.
    """
    if isinstance(cols, pd.MultiIndex) and cols.nlevels >= 2:
        return cols

    new = []
    for c in cols:
        # if c is a tuple already
        if isinstance(c, tuple) and len(c) == 2:
            new.append((str(c[0]), str(c[1])))
            continue

        s = str(c)
        m = _tuple_pat.match(s)
        if m:
            new.append((m.group(1).strip(), m.group(2).strip()))
        else:
            # id columns or other single headers
            new.append((s.strip(), ""))
    return pd.MultiIndex.from_tuples(new, names=["metric", "backend"])

def parse_speedup_csv(path: str) -> pd.DataFrame:
    """
    Returns LONG df with columns:
      conv_type, dataset, heads, feature_dim, metric, backend, value
    Works for:
      A) true 2-row header CSV (metric/backend)
      B) flattened header where columns are "(metric, backend)" strings
    """
    # Try reading as a 2-row header first.
    try:
        df = pd.read_csv(path, header=[0, 1])
        if not (isinstance(df.columns, pd.MultiIndex) and df.columns.nlevels >= 2):
            raise ValueError("not a multiindex header")
    except Exception:
        df = pd.read_csv(path)  # fallback (flattened)

    # Ensure we have MultiIndex columns of shape (metric, backend)
    df.columns = _maybe_make_multiindex(df.columns)

    # Normalize id columns: they should be (name, "").
    # Some pandas versions create ("conv_type","Unnamed: ...") etc; fix by first-level name.
    # Assume first 4 columns correspond to IDs in order.
    cols = list(df.columns)
    for i, name in enumerate(ID_COLS):
        cols[i] = (name, "")
    df.columns = pd.MultiIndex.from_tuples(cols, names=["metric", "backend"])

    # Extract ids
    df_id = df.loc[:, [(c, "") for c in ID_COLS]].copy()
    df_id.columns = ID_COLS
    df_id["heads"] = pd.to_numeric(df_id["heads"], errors="coerce").astype("Int64")
    df_id["feature_dim"] = pd.to_numeric(df_id["feature_dim"], errors="coerce").astype("Int64")

    # Values part
    df_vals = df.drop(columns=[(c, "") for c in ID_COLS])
    # Keep only metrics we need
    df_vals = df_vals[[c for c in df_vals.columns if c[0] in METRICS_KEEP]].copy()
    df_vals.columns.names = ["metric", "backend"]

    # LONG via stack (safe on old pandas)
    long = (
        df_vals
        .stack(level=["metric", "backend"])
        .rename("value")
        .reset_index()
    )
    # stack creates columns: index + metric + backend + value
    # Join ids back using the row index
    long = long.merge(df_id.reset_index().rename(columns={"index": "level_0"}),
                      on="level_0", how="left")

    # Reorder / clean
    long = long.rename(columns={"level_0": "_row"})
    long = long[["_row", "conv_type", "dataset", "heads", "feature_dim", "metric", "backend", "value"]]
    long["value"] = pd.to_numeric(long["value"], errors="coerce")
    return long


In [33]:
CONV_TO_LATEX = {
    "gat_v2": "GATv2",
    "gt": "Graph Transformer",
    "min_aggr": ...,
    "gcn": "SpMM (with GCN as the operator)",
    

}

BACKEND_TO_LATEX = {
    "cusparse": "cuSPARSE",
    "cusparse_precomputed_bwd": "cuSPARSE+ $\\tilde{A^\\top}$)",
    "fusegnn": "FuseGNN",
    "cuda": "Ours",
    "tcgnn": "TC-GNN",
    "triton_block_sparse": "Ours (WSB/TC)",
    "pyg": "PyG",
    "torch_native_gcn": "Pytorch CSR-SpMM",
    "dfgnn": "DF-GNN",
    "cugraph": "cuGraph",
    "dgl": "DGL"
}

In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import re

# ----------------------------
# Robust CSV parser -> LONG

METRICS_KEEP = ["Fwd time", "Bwd time", "Fwd mem", "Bwd mem"]
ID_COLS = ["conv_type", "dataset", "heads", "feature_dim"]
_tuple_pat = re.compile(r"^\(\s*(.+?)\s*,\s*(.+?)\s*\)$")

def _maybe_make_multiindex(cols):
    # If cols are strings like "(Bwd mem, cuda)" -> convert to MultiIndex (metric, backend).
    if isinstance(cols, pd.MultiIndex) and cols.nlevels >= 2:
        return cols
    new = []
    for c in cols:
        if isinstance(c, tuple) and len(c) == 2:
            new.append((str(c[0]).strip(), str(c[1]).strip()))
            continue
        s = str(c)
        m = _tuple_pat.match(s)
        if m:
            new.append((m.group(1).strip(), m.group(2).strip()))
        else:
            new.append((s.strip(), ""))
    return pd.MultiIndex.from_tuples(new, names=["metric", "backend"])

def parse_speedup_csv(path: str) -> pd.DataFrame:
    # Try proper 2-row header first
    try:
        df = pd.read_csv(path, header=[0, 1])
    except Exception:
        df = pd.read_csv(path)

    df.columns = _maybe_make_multiindex(df.columns)

    # Force first 4 columns to be IDs (robustly)
    cols = list(df.columns)
    for i, name in enumerate(ID_COLS):
        cols[i] = (name, "")
    df.columns = pd.MultiIndex.from_tuples(cols, names=["metric", "backend"])

    df_id = df.loc[:, [(c, "") for c in ID_COLS]].copy()
    df_id.columns = ID_COLS
    df_id["heads"] = pd.to_numeric(df_id["heads"], errors="coerce").astype("Int64")
    df_id["feature_dim"] = pd.to_numeric(df_id["feature_dim"], errors="coerce").astype("Int64")

    df_vals = df.drop(columns=[(c, "") for c in ID_COLS])
    # keep only desired metrics
    df_vals = df_vals[[c for c in df_vals.columns if c[0] in METRICS_KEEP]]
    df_vals.columns.names = ["metric", "backend"]

    # Stack into long
    long = (
        df_vals.stack(level=["metric", "backend"])
              .rename("value")
              .reset_index()
    )
    # Merge IDs back by row index
    long = long.merge(df_id.reset_index().rename(columns={"index": "level_0"}), on="level_0", how="left")

    # Clean columns
    cols = list(long.columns)
    # expect: level_0, metric, backend, value, conv_type, dataset, heads, feature_dim
    long = long.rename(columns={
        cols[1]: "metric",
        cols[2]: "backend",
        cols[3]: "value"
    })
    long["value"] = pd.to_numeric(long["value"], errors="coerce")
    return long[["conv_type", "dataset", "heads", "feature_dim", "metric", "backend", "value"]]

# ----------------------------
# LaTeX helpers
# ----------------------------
def latex_escape(s: str) -> str:
    # s = str(s)
    # return (s.replace("\\", r"\textbackslash{}")
    #          .replace("_", r"\_")
    #          .replace("%", r"\%")
    #          .replace("&", r"\&")
    #          .replace("#", r"\#")
    #          .replace("{", r"\{")
    #          .replace("}", r"\}")
    #          .replace("$", r"\$")
    #          .replace("~", r"\textasciitilde{}")
    #          .replace("^", r"\textasciicircum{}"))
    return s

def fmt_num(x, backend=None, ndigits=2):
    # print(backend)
    if pd.isna(x):
        if backend == "dfgnn":
            return "\\texttt{N/A}"
        return "\\texttt{OOM}"

    x = float(x)
    if x != 0.0 and abs(x) < 1e-3:
        return f"{x:.2e}"
    return f"{x:.{ndigits}f}"

def to_latex_multilevel_table(
    piv: pd.DataFrame,
    backends: list[str],
    metrics: list[str],
    caption: str,
    label: str,
    table_star: bool = False,
    resize_to_textwidth: bool = True,
    ndigits: int = 2,
    landscape: bool = False,
) -> str:
    """
    piv: index=dataset, columns=MultiIndex(metric, backend)
    """
    nB = len(backends)


    # Ensure full column grid exists (metric x backend)
    full_cols = pd.MultiIndex.from_product([metrics, backends], names=["metric", "backend"])
    for c in full_cols:
        if c not in piv.columns:
            piv[c] = np.nan
    piv = piv[full_cols]  # ordered

    # Alignment string: l + for each metric: nB columns
    # Example: l ccc ccc ccc ccc (with vertical separators between metric groups)
    group_sep = "|"  # you can change to "" if you don't want vertical lines
    colspec = "l"
    for mi, _m in enumerate(metrics):
        colspec += group_sep + ("c" * nB)

    # Header row 1: metric blocks
    # Header row 2: backend names repeated
    # Compute cmidrule ranges
    start = 2  # because column 1 is Dataset
    cmidrules = []
    metric_hdr = ["Dataset"]
    for m in metrics:
        metric_hdr.append(rf"\multicolumn{{{nB}}}{{c}}{{{latex_escape(m)}}}")
        cmidrules.append((start, start + nB - 1))
        start += nB

    # Second header: backend names
    backend_hdr = [""] + [latex_escape(BACKEND_TO_LATEX[b]) for _m in metrics for b in backends]

    lines = []
    lines.append(r"\toprule")
    lines.append(" & ".join(metric_hdr) + r" \\")
    # cmidrules
    for a, b in cmidrules:
        lines.append(rf"\cmidrule(lr){{{a}-{b}}}")
    lines.append(" & ".join(backend_hdr) + r" \\")
    lines.append(r"\midrule")

    # Body
    for ds in piv.index:
        row = piv.loc[ds]
        vals = [fmt_num(row[(m, b)], backend=b, ndigits=ndigits) for m in metrics for b in backends]
        lines.append(rf"\texttt{{{latex_escape(ds)}}} & " + " & ".join(vals) + r" \\")
    lines.append(r"\bottomrule")

    tabular = rf"\begin{{tabular}}{{{colspec}}}" + "\n" + "\n".join(lines) + "\n" + r"\end{tabular}"

    if resize_to_textwidth:
        tabular = r"\resizebox{\textwidth}{!}{" + "\n" + tabular + "\n" + "}"

    if landscape:
        env = "sidewaystable*" if table_star else "sidewaystable"
    else:
        env = "table*" if table_star else "table"

    if "OOM" in tabular:
        print("GOYDA")
        caption += " \\texttt{OOM} indicates that the we couldn't measure the metric for the layer."
    if "DF-GNN" in tabular:
        print("GOYDA 2")
        caption += f" \\texttt{{N/A}} for {BACKEND_TO_LATEX['dfgnn']} means it encountered illegal memory access on this setup and failed to launch. For our measurements, it indicates that measurements are failed due to OOM of the independent layers"


    tex = rf"""\begin{{{env}}}
\centering
\setlength{{\tabcolsep}}{{4pt}}
{tabular}
\caption{{{caption}}}
\label{{{label}}}
\end{{{env}}}
"""
    return tex

# ----------------------------
# Main generator
# ----------------------------
def generate_tables_per_setup(
    csv_path: str,
    out_dir: str,
    backends_order: list[str] | None = None,
    metrics_order: list[str] = METRICS_KEEP,
    table_star: bool = False,
    resize_to_textwidth: bool = True,
    ndigits: int = 2,
    landscape: bool = False,
    raw: bool = False,
):
    df = parse_speedup_csv(csv_path)

    Path(out_dir).mkdir(parents=True, exist_ok=True)

    # Backends to show
    avail_backends = sorted(df["backend"].dropna().unique().tolist())
    if backends_order is None:
        backends = avail_backends
    else:
        backends = [b for b in backends_order if b in avail_backends]

    # Iterate setups
    setups = (
        df[["conv_type", "heads", "feature_dim"]]
        .dropna()
        .drop_duplicates()
        .sort_values(["conv_type", "heads", "feature_dim"])
        .itertuples(index=False)
    )

    master = []
    for conv_type, heads, feat in setups:
        sub = df[
            (df["conv_type"] == conv_type) &
            (df["heads"] == heads) &
            (df["feature_dim"] == feat) &
            (df["metric"].isin(metrics_order)) &
            (df["backend"].isin(backends))
        ].copy()

        # pivot: dataset x (metric, backend)

        piv = sub.pivot_table(index="dataset", columns=["metric", "backend"], values="value", aggfunc="first")
        piv = piv.sort_index()

        suffix = "higher is better" if not raw else "lower is better"
        if len(metrics_order) == 4:
            prefix = "Forward/Backward latency speedup ($t_{\\text{DGL}}/t_{\\text{backend}}$) and Forward/Backward memory footprint improvement ratio ($\\text{memory}_{\\text{DGL}}/\\text{memory}_{\\text{backend}}$)" if not raw else "Latency (ms) and Memory footprint (MB) measurements"
        elif len(metrics_order) == 2 and metrics_order[0] == "Fwd time":
            prefix = "Forward/Backward latency speedup ($t_{\\text{DGL}}/t_{\\text{backend}}$)" if not raw else "Latency (ms) measurements"
        elif len(metrics_order) == 2 and metrics_order[0] == "Fwd mem":
            prefix = "Forward/Backward memory footprint improvement ratio ($\\text{memory}_{\\text{DGL}}/\\text{memory}_{\\text{backend}}$)" if not raw else "Memory footprint (MB) measurements"
        else:
            assert False

        if conv_type == "gat_v2":
            caption = f"{prefix} on Gatv2 results on all datasets (heads $H$={int(heads)}, head dim $D$={int(feat)})"
        elif conv_type == "gcn":
            caption = f"{prefix} on SpMM (with GCN as the operator) on all datasets (hidden dim $D$={int(feat)})"
        elif conv_type == "min_aggr":
            caption = f"{prefix} on reduction-based convolutions (with $\min$-aggregation as the operator) on all datasets (hidden dim $D$={int(feat)})"
        elif conv_type == "gt":
            caption = f"{prefix} on Graph Transformer on all datasets (heads $H$={int(heads)}, head dim $D$={int(feat) // int(heads)})"
        else:
            assert False
        if suffix != "":
            caption += f"; \\textbf{{{suffix}}}."
        else:
            caption += "."

        metrics_sublabel = '_'.join(('_'.join(metric.split(' ')) for metric in metrics_order)).lower()
        label = f"tab:{conv_type}_h{int(heads)}_d{int(feat)}_" + metrics_sublabel
        if raw:
            label += "_raw"

        tex = to_latex_multilevel_table(
            piv=piv,
            backends=backends,
            metrics=metrics_order,
            caption=latex_escape(caption),
            label=latex_escape(label),
            table_star=table_star,
            resize_to_textwidth=resize_to_textwidth,
            ndigits=ndigits,
            landscape=landscape,
        )

        fname = f"{conv_type}_h{int(heads)}_d{int(feat)}_{metrics_sublabel}.tex"
        fpath = Path(out_dir) / fname
        fpath.write_text(tex)
        master.append(rf"\input{{{fpath.as_posix()}}}")
    try:
        (Path(out_dir) / "ALL_SETUPS.tex").write_text((Path(out_dir) / "ALL_SETUPS.tex").read_text() + "\n" + "\n".join(master))
    except FileNotFoundError:
        (Path(out_dir) / "ALL_SETUPS.tex").write_text("\n".join(master))

    print(f"[OK] wrote {len(master)} setup tables to {out_dir}")
    print(f"[OK] master include: {Path(out_dir) / 'ALL_SETUPS.tex'}")
    print(f"[INFO] backends: {backends}")

# Example:
# generate_tables_per_setup("gt_speedup.csv", out_dir="appendix_tables/gt_setups",
#                           backends_order=["cuda", "dfgnn", "triton_block_sparse"],
#                           table_star=True, resize_to_textwidth=True)


In [ ]:
generate_tables_per_setup("gcn_speedup.csv",   out_dir="appendix_tables/gcn_speedups",
                    # choose only the backends you want to show in appendix, or None for all
                    backends_order=["dfgnn", "cusparse", "cusparse_precomputed_bwd", "fusegnn", "cuda", "tcgnn", "triton_block_sparse", "pyg", "torch_native_gcn"],
                    landscape=False,
                    resize_to_textwidth=True,
                    metrics_order=METRICS_KEEP[:-2]
                    )

generate_tables_per_setup("gcn_speedup.csv",   out_dir="appendix_tables/gcn_speedups",
                    # choose only the backends you want to show in appendix, or None for all
                    backends_order=["dfgnn", "cusparse", "cusparse_precomputed_bwd", "fusegnn", "cuda", "tcgnn", "triton_block_sparse", "pyg", "torch_native_gcn"],
                    landscape=False,
                    resize_to_textwidth=True,
                    metrics_order=METRICS_KEEP[-2:]
                    )



generate_tables_per_setup("min_aggr_speedup.csv",   out_dir="appendix_tables/min_aggr_speedups",
                                              resize_to_textwidth=True,
                    # choose only the backends you want to show in appendix, or None for all
                    backends_order=["cugraph", "dfgnn","cusparse", "cusparse_precomputed_bwd", "fusegnn", "cuda", "tcgnn", "triton_block_sparse", "pyg", "torch_native_gcn"],
                    )


generate_tables_per_setup("gatv2_speedup.csv",   out_dir="appendix_tables/gatv2_speedups",
                                              resize_to_textwidth=False,
                    backends_order=["dfgnn", "cusparse", "cusparse_precomputed_bwd", "fusegnn", "cuda", "tcgnn", "triton_block_sparse", "pyg", "torch_native_gcn"],
                    )


generate_tables_per_setup("gt_speedup.csv",   out_dir="appendix_tables/gt_speedups",
                                              resize_to_textwidth=True,
                    backends_order=["dfgnn","cusparse", "cusparse_precomputed_bwd", "fusegnn", "cuda", "tcgnn", "triton_block_sparse", "pyg", "torch_native_gcn"],
                    )



In [ ]:
gcn_raw = build_raw_table_with_dataset_index(df[df[CONV_COL] == "gcn"])
min_aggr_raw = build_raw_table_with_dataset_index(df[df[CONV_COL] == "min_aggr"])
gatv2_raw = build_raw_table_with_dataset_index(df[df[CONV_COL] == "gat_v2"])
gt_raw = build_raw_table_with_dataset_index(df[df[CONV_COL] == "gt"])



gcn_raw.to_csv("gcn_raw.csv")
min_aggr_raw.to_csv("min_aggr_raw.csv")
gatv2_raw.to_csv("gatv2_raw.csv")
gt_raw.to_csv("gt_raw.csv")


In [ ]:
generate_tables_per_setup("gcn_raw.csv",   out_dir="appendix_tables/gcn_raw",
                                              resize_to_textwidth=True,
                    backends_order=["dgl", "dfgnn", "cusparse", "cusparse_precomputed_bwd", "fusegnn", "cuda", "tcgnn", "triton_block_sparse", "pyg", "torch_native_gcn"],
                    landscape=False,
                    raw=True,
                    metrics_order=METRICS_KEEP[:-2]

                    )

generate_tables_per_setup("gcn_raw.csv",   out_dir="appendix_tables/gcn_raw",
                                              resize_to_textwidth=True,
                    backends_order=["dgl", "dfgnn", "cusparse", "cusparse_precomputed_bwd", "fusegnn", "cuda", "tcgnn", "triton_block_sparse", "pyg", "torch_native_gcn"],
                    landscape=False,
                    raw=True,
                    metrics_order=METRICS_KEEP[-2:]
                    )





generate_tables_per_setup("min_aggr_raw.csv",   out_dir="appendix_tables/min_aggr_raw",
                                              resize_to_textwidth=True,
                    backends_order=["dgl", "cugraph", "dfgnn","cusparse", "cusparse_precomputed_bwd", "fusegnn", "cuda", "tcgnn", "triton_block_sparse", "pyg", "torch_native_gcn"],
                                        raw=True,

                    )


generate_tables_per_setup("gatv2_raw.csv",   out_dir="appendix_tables/gatv2_raw",
                                              resize_to_textwidth=True,
                    backends_order=["dgl", "dfgnn", "cusparse", "cusparse_precomputed_bwd", "fusegnn", "cuda", "tcgnn", "triton_block_sparse", "pyg", "torch_native_gcn"],
                                        raw=True,

                    )


generate_tables_per_setup("gt_raw.csv",   out_dir="appendix_tables/gt_raw",
                                              resize_to_textwidth=True,
                    backends_order=["dgl", "dfgnn","cusparse", "cusparse_precomputed_bwd", "fusegnn", "cuda", "tcgnn", "triton_block_sparse", "pyg", "torch_native_gcn"],
                                        raw=True,

                    )



## GT

In [ ]:
print(build_speedup_table_with_dataset_index(no_reordering_df[no_reordering_df[CONV_COL] == "gt"], "dgl").to_markdown(floatfmt=".2f"))

In [ ]:
print(build_raw_table_with_dataset_index(no_reordering_df[no_reordering_df[CONV_COL] == "gt"]).to_markdown(floatfmt=".2f"))

In [ ]:
print(build_raw_table_with_dataset_index(df[df[CONV_COL] == "gt"], [CONV_COL, DATASET_COL, HEADS_COL, FEATURE_DIM_COL, REORDERING_COL]).to_markdown(floatfmt=".2f"))

## GATv2

In [ ]:
print(build_speedup_table_with_dataset_index(no_reordering_df[(no_reordering_df[CONV_COL] == "gat_v2") & (no_reordering_df[BACKEND_COL] != "pyg")], "dgl").to_markdown(floatfmt=".2f"))

In [ ]:
print(build_raw_table_with_dataset_index(no_reordering_df[(no_reordering_df[CONV_COL] == "gat_v2") & (no_reordering_df[BACKEND_COL] != "pyg")]).to_markdown(floatfmt=".2f"))

## GCN

In [ ]:
print(build_speedup_table_with_dataset_index(no_reordering_df[no_reordering_df[CONV_COL] == "gcn"], "dgl").to_markdown(floatfmt=".2f"))

In [ ]:
print(build_raw_table_with_dataset_index(no_reordering_df[no_reordering_df[CONV_COL] == "gcn"]).to_markdown(floatfmt=".2f"))

In [ ]:
print(build_raw_table_with_dataset_index(df[df[CONV_COL] == "gcn"], [CONV_COL, DATASET_COL, HEADS_COL, FEATURE_DIM_COL, REORDERING_COL]).to_markdown(floatfmt=".2f"))

## MinAggr

In [ ]:
print(build_speedup_table_with_dataset_index(no_reordering_df[no_reordering_df[CONV_COL] == "min_aggr"], "dgl").to_markdown(floatfmt=".2f"))

In [ ]:
print(build_speedup_table_with_dataset_index(no_reordering_df[no_reordering_df[CONV_COL] == "min_aggr"], "dgl").to_markdown(floatfmt=".2f"))

In [ ]:
print(build_speedup_table_with_dataset_index(no_reordering_df[no_reordering_df[CONV_COL] == "min_aggr"], "dgl").to_markdown(floatfmt=".2f"))

In [ ]:
print(build_raw_table_with_dataset_index(no_reordering_df[no_reordering_df[CONV_COL] == "min_aggr"]).to_markdown(floatfmt=".2f"))

In [ ]:
zf

In [ ]:
df_test = pd.read_csv("../test_partition", index_col=0)

In [ ]:
df_test

In [ ]:
df_test["graph_reordering_partition_size"] = df_test["graph_reordering_partition_size"].astype(str)

In [ ]:
for dataset in df_test[DATASET_COL].unique():
    df_dataset = df_test[df_test[DATASET_COL] == dataset]
    fig = px.scatter(df_dataset, x="graph_reordering_partition_size", y=FWD_MS_COL, title=dataset, color=BACKEND_COL)
    fig.show()

In [ ]:
import pandas as pd

In [ ]:
gatv2_old = pd.concat([
    pd.read_csv("/home/fvelikon/projects/cuda_exp/out/gat_v2_cuda_new_1", index_col=0),
    pd.read_csv("/home/fvelikon/projects/cuda_exp/out/gat_v2_cuda_new_2", index_col=0),
    pd.read_csv("/home/fvelikon/projects/cuda_exp/out/gat_v2_cuda_new_3", index_col=0),
    ])
gatv2_new = pd.read_csv("/home/fvelikon/projects/cuda_exp/out/gatv2_shmem_full", index_col=0)

gatv2_old

In [ ]:
diff_df = pd.merge(gatv2_old, gatv2_new, on=[DATASET_COL, "feature_dim", "heads", "grad_A_reduce_row_chunk_size"], suffixes=["_old", "_new"])

diff_df["fwd_diff"] = diff_df["forward_ms_old"] - diff_df["forward_ms_new"]
diff_df["bwd_diff"] = diff_df["backward_ms_old"] - diff_df["backward_ms_new"]


diff_df["fwd_rel_diff"] = (diff_df["forward_ms_old"] - diff_df["forward_ms_new"] )/ diff_df["forward_ms_old"]
diff_df["bwd_rel_diff"] = (diff_df["backward_ms_old"] - diff_df["backward_ms_new"]) / diff_df["backward_ms_old"]

diff_df["total_rel_diff"] = (diff_df["forward_ms_old"] - diff_df["forward_ms_new"] + diff_df["backward_ms_old"] - diff_df["backward_ms_new"]) / (diff_df["forward_ms_old"] + diff_df["backward_ms_old"])

In [ ]:
diff_df[[DATASET_COL, "feature_dim", "heads", "grad_A_reduce_row_chunk_size", "fwd_diff", "bwd_diff", "fwd_rel_diff", "bwd_rel_diff", "total_rel_diff"]]

In [ ]:
diff_df.groupby(DATASET_COL)[["fwd_rel_diff", "bwd_rel_diff", "total_rel_diff"]].mean()

In [ ]:
reordering_test_df = pd.read_csv("../out/reordering_test_dot_aggregation_benchmark").loc[
    :, [DATASET_COL, FWD_MS_COL, FEATURE_DIM_COL, "kernel_kind", "use_second_access", "use_vectorized_loads", "graph_reordering_partition_size"]
]
reordering_test_df = reordering_test_df[reordering_test_df["use_second_access"] == True]

reordering_test_df["kernel_kind"] = reordering_test_df["kernel_kind"].map({0: "Feature-coalesced", 1: "Neighbor-coalesced"})
# reordering_test_df["use_second_access"] = reordering_test_df["use_second_access"].map({False: "Single pass", True: "Double pass"})
reordering_test_df["use_vectorized_loads"] = reordering_test_df["use_vectorized_loads"].map({False: "1 load", True: "Vectorized loads"})
reordering_test_df

In [ ]:
reordering_test_df = reordering_test_df[
    ~reordering_test_df[DATASET_COL].isin([
        "web-fraud",
        "web-topics",
        "artnet-views",
        "hm-prices"
    ])
]

In [ ]:
p = pd.pivot(reordering_test_df, index=[DATASET_COL, FEATURE_DIM_COL, "kernel_kind", "use_vectorized_loads"], columns=["graph_reordering_partition_size"], values=FWD_MS_COL)

p_ratio = p.rdiv(p[-1], axis=0)

p0 = p_ratio.xs("Feature-coalesced", level="kernel_kind")  # index: (feature_dim, use_vectorized_loads, use_second_access)
p1 = p_ratio.xs("Neighbor-coalesced", level="kernel_kind")

p_diff = p0 - p1   # same shape as p0/p1: rows=(feature_dim, use_vectorized_loads, use_second_access), cols=partition_size
p_diff

p_rel = p0 / p1          # >1 means kernel 0 has larger speedup
p_pct = (p0 / p1 - 1.0)  # fractional advantage (e.g., 0.2 = +20%)

In [ ]:
# p_pct

In [ ]:
# p_pct.loc["pubmed"]

In [ ]:
# p_pct.to_csv("p_pct.csv")

In [ ]:
# p_diff.to_csv("p_diff.csv")

In [ ]:
# p1.to_csv("p1.csv")

In [ ]:
# p_diff

In [ ]:
p_diff

In [ ]:
p_diff_unpivot = p_diff.reset_index().melt(
    id_vars=['dataset', 'feature_dim', 'use_vectorized_loads'],
    var_name='partition_size',
    value_name='value'
)

p_diff_unpivot = p_diff_unpivot[(p_diff_unpivot["partition_size"] != -1)]
p_diff_unpivot["value"] *= -1

In [ ]:
p_diff_unpivot["feature_dim"] = p_diff_unpivot["feature_dim"].astype(str)

In [ ]:
# px.box(p_diff_unpivot, y="value", x="feature_dim", color="dataset")

In [ ]:
fig = px.box(p_diff_unpivot[p_diff_unpivot["use_vectorized_loads"] != "Vectorized loads"], y="value", x="dataset", color="feature_dim", title="Delta s, NO vectorized loads")
fig.update_layout(yaxis_range=[-1.05,1.4])

In [ ]:
fig = px.box(p_diff_unpivot[p_diff_unpivot["use_vectorized_loads"] == "Vectorized loads"], y="value", x="dataset", color="feature_dim", title="Delta s, USE vectorized loads")
fig.update_layout(yaxis_range=[-1.05,1.4])

In [ ]:
datasets_to_present = [
    "ogbn-products",
    "pokec-regions",
    "web-traffic",
    "artnet-exp",
    "city-roads-L",
]

In [ ]:
p_diff_unpivot_slice = p_diff_unpivot[p_diff_unpivot["dataset"].isin(datasets_to_present)]

In [ ]:
fig = px.box(p_diff_unpivot_slice[p_diff_unpivot_slice["use_vectorized_loads"] != "Vectorized loads"], y="value", x="dataset", color="feature_dim", title="Delta s, NO vectorized loads")
fig.update_layout(yaxis_range=[-1.05,1.4])

In [ ]:
fig = px.box(p_diff_unpivot_slice[p_diff_unpivot_slice["use_vectorized_loads"] == "Vectorized loads"], y="value", x="dataset", color="feature_dim", title="Delta s, USE vectorized loads")
fig.update_layout(yaxis_range=[-1.05,1.4])

In [ ]:
datasets_to_present = [
    "artnet-exp",
    "ogbn-products",
    "pokec-regions",
    "city-roads-L",
    "web-traffic",
]

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

plt.rcParams['text.usetex'] = True
plt.rcParams['font.family'] = 'serif'

# Красивая палитра (похожая на plotly)
colors = {
    "32": '#636EFA',   # синий
    "64": '#EF553B',   # красный/оранжевый
    "128": '#00CC96',  # зелёный
    "256": '#AB63FA',  # фиолетовый
    "512": '#FFA15A',  # оранжевый
}

datasets = datasets_to_present
feature_dims = sorted(p_diff_unpivot_slice['feature_dim'].unique())

fig, ax = plt.subplots(figsize=(14, 6))

# Позиции для боксов
n_datasets = len(datasets)
n_features = len(feature_dims)
group_width = 0.8

box_width = group_width / (n_features * 2 + 1)

positions_all = []
colors_all = []
hatches_all = []
data_all = []

for i, dataset in enumerate(datasets):
    base_pos = i
    for j, fd in enumerate(feature_dims):
        # Позиция для пары (no vectorized, vectorized)
        pos_no_vec = base_pos - group_width/2 + (2*j + 0.5) * box_width
        pos_vec = base_pos - group_width/2 + (2*j + 1.5) * box_width
        
        # Данные
        data_no_vec = p_diff_unpivot_slice[
            (p_diff_unpivot_slice['dataset'] == dataset) & 
            (p_diff_unpivot_slice['feature_dim'] == fd) & 
            (p_diff_unpivot_slice['use_vectorized_loads'] != 'Vectorized loads')
        ]['value'].values
        
        data_vec = p_diff_unpivot_slice[
            (p_diff_unpivot_slice['dataset'] == dataset) & 
            (p_diff_unpivot_slice['feature_dim'] == fd) & 
            (p_diff_unpivot_slice['use_vectorized_loads'] == 'Vectorized loads')
        ]['value'].values
        
        if len(data_no_vec) > 0:
            positions_all.append(pos_no_vec)
            colors_all.append(colors[fd])
            hatches_all.append(None)  # сплошная заливка
            data_all.append(data_no_vec)
        
        if len(data_vec) > 0:
            positions_all.append(pos_vec)
            colors_all.append(colors[fd])
            hatches_all.append('///')  # полоски
            data_all.append(data_vec)

# Рисуем боксплоты по одному для контроля над стилями
for pos, data, color, hatch in zip(positions_all, data_all, colors_all, hatches_all):
    bp = ax.boxplot([data], positions=[pos], widths=box_width * 0.9,
                    patch_artist=True, manage_ticks=False)
    
    for patch in bp['boxes']:
        patch.set_facecolor(color)
        patch.set_alpha(0.7 if hatch is None else 0.4)
        if hatch:
            patch.set_hatch(hatch)
        patch.set_edgecolor('black')
        patch.set_linewidth(1)
    
    for element in ['whiskers', 'caps']:
        for line in bp[element]:
            line.set_color('black')
            line.set_linewidth(1)
    
    for line in bp['medians']:
        line.set_color('black')
        line.set_linewidth(1.5)
    
    for flier in bp['fliers']:
        flier.set(marker='o', markersize=4, alpha=0.6)

# Настройка осей
ax.set_xticks(range(n_datasets))
ax.set_xticklabels([rf'\texttt{{{d}}}' for d in datasets], fontsize=18)
# ax.set_xlabel('Dataset', fontsize=12)
ax.set_ylabel(r'$\Delta s$', fontsize=14)
# ax.set_title(r'$\Delta s$: Vectorized vs Non-vectorized loads', fontsize=14, fontweight='bold')

# Легенда
legend_elements = []
for i, fd in enumerate(sorted(feature_dims, key=lambda x: int(x))):
    # Сплошной
    # legend_elements.append(mpatches.Patch(facecolor=colors[fd], alpha=0.7, 
    #                                       edgecolor='black', label=f'{fd} (scalar)'))
    legend_elements.insert(i, mpatches.Patch(facecolor=colors[fd], alpha=0.7, edgecolor='black', label=f'$D$={fd} (scalar)'))
    # С полосками
    legend_elements.append(mpatches.Patch(facecolor=colors[fd], alpha=0.4,
                                          edgecolor='black', hatch='///', label=f'$D$={fd} (vec)'))



ax.legend(handles=legend_elements, loc='upper right', ncol=2, fontsize=14)



# Текст + стрелки - сдвинуты левее
ax.text(-0.60, 0.63, r'\texttt{neighbor-parallel} benefits from reordering', 
        fontsize=18, verticalalignment='center', style='italic', color='darkgreen')
ax.text(-0.60, -0.68, r'\texttt{feature-parallel} benefits from reordering', 
        fontsize=18, verticalalignment='center', style='italic', color='darkred')


ax.annotate('', xy=(-0.55, 0.59), xytext=(-0.55, 0.05),
            arrowprops=dict(arrowstyle='-|>', color='darkgreen', lw=2))


ax.annotate('', xy=(-0.55, -0.64), xytext=(-0.55, -0.05),
            arrowprops=dict(arrowstyle='-|>', color='darkred', lw=2))


ax.set_ylim(-1.05, 1.4)
ax.axhline(y=0, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('reorder_delta_main.pdf', bbox_inches='tight')  # можно вставить напрямую в LaTeX\plt.show()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

plt.rcParams['text.usetex'] = True
plt.rcParams['font.family'] = 'serif'

# Красивая палитра (похожая на plotly)
colors = {
    "32": '#636EFA',   # синий
    "64": '#EF553B',   # красный/оранжевый
    "128": '#00CC96',  # зелёный
    "256": '#AB63FA',  # фиолетовый
    "512": '#FFA15A',  # оранжевый
}

datasets = p_diff_unpivot.dataset.unique()

feature_dims = sorted(p_diff_unpivot['feature_dim'].unique())

fig, ax = plt.subplots(figsize=(34, 8))

# Позиции для боксов
n_datasets = len(datasets)
n_features = len(feature_dims)
group_width = 0.8

box_width = group_width / (n_features * 2 + 1)

positions_all = []
colors_all = []
hatches_all = []
data_all = []

for i, dataset in enumerate(datasets):
    base_pos = i
    for j, fd in enumerate(feature_dims):
        # Позиция для пары (no vectorized, vectorized)
        pos_no_vec = base_pos - group_width/2 + (2*j + 0.5) * box_width
        pos_vec = base_pos - group_width/2 + (2*j + 1.5) * box_width
        
        # Данные
        data_no_vec = p_diff_unpivot[
            (p_diff_unpivot['dataset'] == dataset) & 
            (p_diff_unpivot['feature_dim'] == fd) & 
            (p_diff_unpivot['use_vectorized_loads'] != 'Vectorized loads')
        ]['value'].values
        
        data_vec = p_diff_unpivot[
            (p_diff_unpivot['dataset'] == dataset) & 
            (p_diff_unpivot['feature_dim'] == fd) & 
            (p_diff_unpivot['use_vectorized_loads'] == 'Vectorized loads')
        ]['value'].values
        
        if len(data_no_vec) > 0:
            positions_all.append(pos_no_vec)
            colors_all.append(colors[fd])
            hatches_all.append(None)  # сплошная заливка
            data_all.append(data_no_vec)
        
        if len(data_vec) > 0:
            positions_all.append(pos_vec)
            colors_all.append(colors[fd])
            hatches_all.append('///')  # полоски
            data_all.append(data_vec)

# Рисуем боксплоты по одному для контроля над стилями
for pos, data, color, hatch in zip(positions_all, data_all, colors_all, hatches_all):
    bp = ax.boxplot([data], positions=[pos], widths=box_width * 0.9,
                    patch_artist=True, manage_ticks=False)
    
    for patch in bp['boxes']:
        patch.set_facecolor(color)
        patch.set_alpha(0.7 if hatch is None else 0.4)
        if hatch:
            patch.set_hatch(hatch)
        patch.set_edgecolor('black')
        patch.set_linewidth(1)
    
    for element in ['whiskers', 'caps']:
        for line in bp[element]:
            line.set_color('black')
            line.set_linewidth(1)
    
    for line in bp['medians']:
        line.set_color('black')
        line.set_linewidth(1.5)
    
    for flier in bp['fliers']:
        flier.set(marker='o', markersize=4, alpha=0.6)

# Настройка осей
ax.set_xticks(range(n_datasets))
ax.set_xticklabels([rf'\texttt{{{d}}}' for d in datasets], fontsize=18, rotation=45, ha='right')
# ax.set_xlabel('Dataset', fontsize=12)
ax.set_ylabel(r'$\Delta s$', fontsize=14)
# ax.set_title(r'$\Delta s$: Vectorized vs Non-vectorized loads', fontsize=14, fontweight='bold')

# Легенда
legend_elements = []
for i, fd in enumerate(sorted(feature_dims, key=lambda x: int(x))):
    # Сплошной
    # legend_elements.append(mpatches.Patch(facecolor=colors[fd], alpha=0.7, 
    #                                       edgecolor='black', label=f'{fd} (scalar)'))
    legend_elements.insert(i, mpatches.Patch(facecolor=colors[fd], alpha=0.7, edgecolor='black', label=f'$D$={fd} (scalar)'))
    # С полосками
    legend_elements.append(mpatches.Patch(facecolor=colors[fd], alpha=0.4,
                                          edgecolor='black', hatch='///', label=f'$D$={fd} (vec)'))



ax.legend(handles=legend_elements, loc='upper right', ncol=2, fontsize=14)



# Текст + стрелки - сдвинуты левее
ax.text(7.50, 0.63, r'\texttt{neighbor-parallel} benefits from reordering', 
        fontsize=18, verticalalignment='center', style='italic', color='darkgreen')
ax.text(7.50, -0.68, r'\texttt{feature-parallel} benefits from reordering', 
        fontsize=18, verticalalignment='center', style='italic', color='darkred')


ax.annotate('', xy=(7.55, 0.59), xytext=(7.55, 0.05),
            arrowprops=dict(arrowstyle='-|>', color='darkgreen', lw=2))


ax.annotate('', xy=(7.55, -0.64), xytext=(7.55, -0.05),
            arrowprops=dict(arrowstyle='-|>', color='darkred', lw=2))


ax.set_ylim(-1.05, 1.4)
ax.axhline(y=0, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('reorder_delta_full_appendix.pdf', bbox_inches='tight')  # можно вставить напрямую в LaTeX\plt.show()
plt.show()

# check the difference in measurements

In [ ]:
baselines = pd.merge(
    pd.read_csv("../out/gatv2_new_sync", index_col=0),
    gatv2_df[gatv2_df[BACKEND_COL] != "cuda"],
    on=[DATASET_COL, BACKEND_COL, FEATURE_DIM_COL, HEADS_COL, "graph_reordering_partition_size"],
    suffixes=["_new", "_old"]
)[[DATASET_COL, BACKEND_COL, FEATURE_DIM_COL, HEADS_COL, "forward_ms_new", "forward_ms_old"]]


cuda = pd.merge(
    pd.read_csv("../out/gatv2_new_sync", index_col=0),
    gatv2_df[gatv2_df[BACKEND_COL] == "cuda"],
    on=[DATASET_COL, BACKEND_COL, FEATURE_DIM_COL, HEADS_COL, "grad_A_reduce_row_chunk_size", "graph_reordering_partition_size"],
    suffixes=["_new", "_old"]
)[[DATASET_COL, BACKEND_COL, FEATURE_DIM_COL, HEADS_COL, "forward_ms_new", "forward_ms_old"]]



diff_df = pd.concat([baselines, cuda]).sort_values(by=[DATASET_COL, BACKEND_COL, FEATURE_DIM_COL, HEADS_COL])